In [ ]:
import os
import globautoriaNumberplateOcrRu-2021-09-01
import math
import random
import json
import glob

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_ROOT = r"Datasets\OCR\autoriaNumberplateOcrRu"

BATCH_SIZE = 32
IMG_HEIGHT = 32
IMG_WIDTH = 128
MAX_EPOCHS = 150
NUM_WORKERS = 0
RANDOM_SEED = 42

MAX_TRAIN_SAMPLES = 3000
MAX_TEST_SAMPLES = 300

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
ALPHABET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
blank_idx = 0
char2idx = {c: i + 1 for i, c in enumerate(ALPHABET)}
idx2char = {i + 1: c for i, c in enumerate(ALPHABET)}
vocab_size = len(ALPHABET) + 1

train_transforms = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

test_transforms = train_transforms

class NumberplateOCRDataset(Dataset):
    def __init__(self, root, split, transforms=None, max_samples=None):
        self.root = root
        self.split = split
        self.transforms = transforms
        ann_dir = os.path.join(root, split, "ann")
        self.ann_paths = glob.glob(os.path.join(ann_dir, "*.json"))
        self.ann_paths.sort()
        if max_samples is not None:
            self.ann_paths = self.ann_paths[:max_samples]

    def __len__(self):
        return len(self.ann_paths)

    def encode_text(self, text):
        return [char2idx[c] for c in str(text) if c in char2idx]

    def __getitem__(self, idx):
        ann_path = self.ann_paths[idx]
        with open(ann_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        name = data["name"]
        text = str(data.get("description", ""))
        img_rel = os.path.join(self.split, "img", name + ".png")
        img_path = os.path.join(self.root, img_rel)
        image = Image.open(img_path).convert("RGB")
        if self.transforms is not None:
            image = self.transforms(image)
        label = torch.tensor(self.encode_text(text), dtype=torch.long)
        return image, label, text

train_dataset = NumberplateOCRDataset(
    DATASET_ROOT, "train", transforms=train_transforms, max_samples=MAX_TRAIN_SAMPLES
)
test_dataset = NumberplateOCRDataset(
    DATASET_ROOT, "test", transforms=test_transforms, max_samples=MAX_TEST_SAMPLES
)

def collate_fn(batch):
    batch = [b for b in batch if len(b[1]) > 0]
    images = [b[0] for b in batch]
    labels = [b[1] for b in batch]
    texts = [b[2] for b in batch]
    images = torch.stack(images, dim=0)
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    targets = torch.cat(labels, dim=0)
    return images, targets, label_lengths, texts

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    drop_last=False
)

In [ ]:

def decode_sequence(seq):
    seq = seq.cpu().numpy().tolist()
    prev = None
    out = []
    for s in seq:
        if s != blank_idx and s != prev:
            if s in idx2char:
                out.append(idx2char[s])
        prev = s
    return "".join(out)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for images, targets, target_lengths, _ in loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(images)
        T_cur, N_cur, C_cur = logits.size()
        input_lengths = torch.full(
            size=(N_cur,), fill_value=T_cur, dtype=torch.long, device=logits.device
        )
        loss = criterion(logits, targets, input_lengths, target_lengths)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item()
    return running_loss / max(1, len(loader))

def evaluate(model, loader, max_batches=3):
    model.eval()
    samples = []
    with torch.no_grad():
        for bi, (images, targets, target_lengths, texts) in enumerate(loader):
            images = images.to(device)
            logits = model(images)
            pred = logits.permute(1, 0, 2).argmax(2)
            for i in range(images.size(0)):
                decoded = decode_sequence(pred[i])
                gt = texts[i]
                samples.append((gt, decoded))
            if bi + 1 >= max_batches:
                break
    return samples

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch}/{MAX_EPOCHS}, train loss: {train_loss:.4f}")

test_samples = evaluate(model, test_loader, max_batches=2)
for i, (gt, pred) in enumerate(test_samples[:20]):
    print(f"{i+1}. GT: {gt} | PRED: {pred}")

In [ ]:
def evaluate_with_metrics(model, loader, max_batches=10, max_print=20):
    model.eval()
    total_seq = 0
    correct_seq = 0
    total_chars = 0
    correct_chars = 0
    samples = []
    with torch.no_grad():
        for bi, (images, targets, target_lengths, texts) in enumerate(loader):
            images = images.to(device)
            logits = model(images)
            pred = logits.permute(1, 0, 2).argmax(2)
            for i in range(images.size(0)):
                decoded = decode_sequence(pred[i])
                gt = str(texts[i])
                samples.append((gt, decoded))
                total_seq += 1
                if decoded == gt:
                    correct_seq += 1
                L = max(len(gt), len(decoded))
                if L > 0:
                    total_chars += L
                    correct_chars += sum(
                        1 for a, b in zip(gt.ljust(L), decoded.ljust(L)) if a == b
                    )
            if bi + 1 >= max_batches:
                break
    seq_acc = correct_seq / max(1, total_seq)
    char_acc = correct_chars / max(1, total_chars)
    print(f"Exact plate accuracy: {seq_acc * 100:.2f}%")
    print(f"Character accuracy:   {char_acc * 100:.2f}%")
    print()
    for i, (gt, pred) in enumerate(samples[:max_print]):
        print(f"{i+1}. GT: {gt} | PRED: {pred}")
    return seq_acc, char_acc, samples

seq_acc, char_acc, test_samples = evaluate_with_metrics(model, test_loader, max_batches=10, max_print=30)